# Real-time Unique Person Re-Identification (Final Stable Version)

Esta versão final foi calibrada para ser **extremamente tolerante** a mudanças de ângulo, luz e movimento, resolvendo o problema de IDs duplicados (#1 e #2 aparecendo para a mesma pessoa).

### Step 1: Requirements
```bash
pip install ultralytics opencv-python torch torchvision scikit-learn ipywidgets
```

In [1]:
import cv2
import torch
import os
import time
import numpy as np
from ultralytics import YOLO
from torchvision import models, transforms
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
device = "cuda" if torch.cuda.is_available() else "cpu"

yolo_model = YOLO("yolo11n.pt")
yolo_model.to(device)

reid_model = models.mobilenet_v3_small(weights='DEFAULT')
reid_model.classifier = torch.nn.Identity()
reid_model.to(device)
reid_model.eval()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"✅ Sistema calibrado no {device}.")

✅ Sistema calibrado no cpu.


### Step 2: Executar Detecção (Calibração Máxima)

**O que mudou para ser mais preciso:**
- **Threshold (0.72):** Mais tolerante para reconhecer você mesmo com sombras ou de lado.
- **Min Estabilidade (20 frames):** Evita que IDs passageiros entrem no contador final.
- **Melhor Matching:** Compara a imagem atual com TODAS as fotos salvas de cada pessoa.

In [ ]:
image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

cap = cv2.VideoCapture(0)

# --- CALIBRAÇÃO ---
person_db = {} 
next_global_id = 1
REID_THRESHOLD = 0.72 # Reduzido para ser mais tolerante (0.70 - 0.75 é ideal)
MIN_STABILITY_FRAMES = 20 # Aumentado para 20 frames (~1-2 seg) para confirmar novo ID
MAX_SAMPLES = 15 # Guardamos até 15 variações visuais por pessoa
# ------------------

prev_time = 0
print("🔄 Contador resetado. Iniciando calibração estável...")

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        # Detecção com confiança maior para evitar falso positivo
        results = yolo_model(frame, verbose=False, classes=[0], conf=0.65)
        annotated_frame = frame.copy()
        
        if results and len(results[0].boxes) > 0:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            
            for box in boxes:
                x1, y1, x2, y2 = map(int, box)
                person_img = frame[max(0, y1):y2, max(0, x1):x2]
                if person_img.size == 0: continue
                
                input_tensor = preprocess(person_img).unsqueeze(0).to(device)
                with torch.no_grad():
                    embedding = reid_model(input_tensor).cpu().numpy().reshape(1, -1)
                
                best_match_id = None
                highest_sim = -1
                
                # Comparação rigorosa
                for pid, data in person_db.items():
                    # Verifica contra todas as assinaturas dessa pessoa
                    for sig in data['signatures']:
                        sim = cosine_similarity(embedding, sig)[0][0]
                        if sim > REID_THRESHOLD and sim > highest_sim:
                            highest_sim = sim
                            best_match_id = pid
                
                if best_match_id is not None:
                    # Atualiza pessoa conhecida
                    person_db[best_match_id]['frames_seen'] += 1
                    # Adiciona nova assinatura se for uma imagem "nova/diferente" da mesma pessoa
                    if highest_sim < 0.90 and len(person_db[best_match_id]['signatures']) < MAX_SAMPLES:
                        person_db[best_match_id]['signatures'].append(embedding)
                else:
                    # Só cria novo ID se realmente não parece com nada
                    best_match_id = next_global_id
                    next_global_id += 1
                    person_db[best_match_id] = {'signatures': [embedding], 'frames_seen': 1}
                
                # UI Visual Estabilizada
                is_stable = person_db[best_match_id]['frames_seen'] >= MIN_STABILITY_FRAMES
                color = (0, 255, 0) if is_stable else (0, 165, 255)
                label = f"ID #{best_match_id}" if is_stable else f"Analisando ({person_db[best_match_id]['frames_seen']}/{MIN_STABILITY_FRAMES})"
                
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(annotated_frame, label, (x1, y1-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # Filtrar total geral apenas por IDs confirmados
        total_count = len([p for p in person_db.values() if p['frames_seen'] >= MIN_STABILITY_FRAMES])

        curr_time = time.time()
        fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
        prev_time = curr_time

        cv2.rectangle(annotated_frame, (0, 0), (300, 100), (0, 0, 0), -1)
        cv2.putText(annotated_frame, f"FPS: {int(fps)}", (20, 35), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(annotated_frame, f"Total Unico: {total_count}", (20, 75), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        _, buffer = cv2.imencode('.jpg', annotated_frame)
        image_widget.value = buffer.tobytes()
        
except KeyboardInterrupt: pass
finally:
    cap.release()
    print(f"Finalizado. Total de pessoas confirmadas: {total_count}")

Image(value=b'', format='jpeg', height='480', width='640')

🔄 Contador resetado. Iniciando calibração estável...
Finalizado. Total de pessoas confirmadas: 2
